In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import gzip
import warnings
warnings.filterwarnings('ignore')

columns = [
    'karl_id', 'host_name', 'model_name', 'hardware_make', 'karl_last_seen',
    'auth_username', 'serial_number', 'group_id', 'tenant_id', 'platform',
    'metric_category', 'measure_name', 'time', 'p90_processor_time',
    'avg_processor_time', 'max_cpu_usage', 'p90_memory_utilization',
    'avg_memory_utilization', 'max_memory_usage', 'p10_battery_health',
    'avg_battery_health', 'cpu_count', 'memory_count', 'memory_size_gb',
    'driver_vendor', 'os', 'wifi_mac_add', 'driver_version', 'driver_date',
    'os_version', 'driver', 'agent_id', 'performance_status', 'device_status',
    'max_battery_temperature', 'avg_battery_temperature', 'p90_battery_temperature',
    'avg_cpu_temp', 'p90_cpu_temp', 'avg_battery_discharge', 'p90_battery_discharge',
    'avg_boot_time', 'p90_boot_time', 'uptime_days', 'total_app_crash'
]

chunk_size = 100000
sample_data = []

with gzip.open('000.gz', 'rt') as f:
    for i, chunk in enumerate(pd.read_csv(f, sep='|', names=columns, chunksize=chunk_size)):
        sample_data.append(chunk)
        if i >= 4:
            break

df = pd.concat(sample_data, ignore_index=True)

numeric_cols = [
    'avg_processor_time', 'max_cpu_usage', 'avg_memory_utilization',
    'max_memory_usage', 'avg_battery_health', 'cpu_count', 'memory_size_gb',
    'avg_cpu_temp', 'avg_boot_time', 'p90_boot_time',
    'uptime_days', 'total_app_crash'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

feature_cols = ['max_cpu_usage','avg_memory_utilization','cpu_count','memory_size_gb','uptime_days','avg_cpu_temp']

model_df = df[feature_cols + ['total_app_crash']].dropna()

threshold = model_df['total_app_crash'].quantile(0.90)
model_df['y'] = (model_df['total_app_crash'] >= threshold).astype(int)

model_df['high_cpu_flag'] = ((model_df["max_cpu_usage"] > 70) & (model_df["max_cpu_usage"] < 100)).astype(int)
model_df['high_mem_flag'] = ((model_df["avg_memory_utilization"] > 70) & (model_df["avg_memory_utilization"] < 90)).astype(int)
model_df['high_temp_flag'] = ((model_df["avg_cpu_temp"] > 70) & (model_df["avg_cpu_temp"] < 95)).astype(int)
model_df['high_uptime_flag'] = ((model_df["uptime_days"] >= 26) & (model_df["uptime_days"] <= 60)).astype(int)

feature_cols = ['max_cpu_usage','avg_memory_utilization','cpu_count','memory_size_gb','uptime_days','avg_cpu_temp',
    'high_cpu_flag','high_mem_flag','high_temp_flag','high_uptime_flag'
]

X = model_df[feature_cols]
y = model_df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200,max_depth=12,class_weight="balanced",random_state=42,n_jobs=-1)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\n CLASSIFICATION PERFORMANCE ")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop Risk Drivers:")
print(importance.head())

new_host = pd.DataFrame([{
    'max_cpu_usage': 98,
    'avg_memory_utilization': 85,
    'cpu_count': 8,
    'memory_size_gb': 16,
    'avg_cpu_temp': 105,
    'uptime_days': 300
}])

new_host['high_cpu_flag'] = ((new_host["max_cpu_usage"] > 70) & (new_host["max_cpu_usage"] < 100)).astype(int)
new_host['high_mem_flag'] = ((new_host["avg_memory_utilization"] > 70) & (new_host["avg_memory_utilization"] < 90)).astype(int)
new_host['high_temp_flag'] = ((new_host["avg_cpu_temp"] > 70) & (new_host["avg_cpu_temp"] < 95)).astype(int)
new_host['high_uptime_flag'] = ((new_host["uptime_days"] >= 26) & (new_host["uptime_days"] <= 60)).astype(int)

new_host = new_host[feature_cols]

pred = model.predict(new_host)[0]
prob = model.predict_proba(new_host)[0, 1]

print("\nNew Host Risk:")
print("High Risk (0/1):", pred)
print("Risk Probability:", prob)


 CLASSIFICATION PERFORMANCE 
              precision    recall  f1-score   support

           0       0.99      0.86      0.92      4099
           1       0.41      0.89      0.56       457

    accuracy                           0.86      4556
   macro avg       0.70      0.87      0.74      4556
weighted avg       0.93      0.86      0.88      4556

ROC-AUC: 0.9331992699292084

Top Risk Drivers:
                  feature  importance
4             uptime_days    0.375740
1  avg_memory_utilization    0.175912
2               cpu_count    0.108268
3          memory_size_gb    0.103821
7           high_mem_flag    0.076060

New Host Risk:
High Risk (0/1): 0
Risk Probability: 0.035


In [18]:
import pandas as pd
import numpy as np

features = ['avg_cpu_temp','max_cpu_usage','avg_memory_utilization','uptime_days', 'memory_size_gb']

threshold_results = []

for feature in features:
    temp = model_df[[feature, "y"]].copy()
    temp = temp.dropna()
    
    # suggestion from ai - use qcut to make sure there are equal numbers of hosts in each bin
    temp["bin"] = pd.qcut(temp[feature], q=10, duplicates="drop")

    risk_table = temp.groupby("bin")["y"].mean() #crash rate in each bin
    
    risk_diff = risk_table.diff() #increases between bins?
    
    jump_idx = risk_diff.idxmax() #bin with highest jump in risk
    
    threshold_results.append({"feature": feature,"threshold_bin": str(jump_idx),"max_risk_increase": risk_diff.max(),
        "min_risk": risk_table.min(),"max_risk": risk_table.max()
    })

threshold_df = pd.DataFrame(threshold_results)

print("\n AUTOMATIC THRESHOLDS (ALL FEATURES) ")
print(threshold_df)


 AUTOMATIC THRESHOLDS (ALL FEATURES) 
                  feature    threshold_bin  max_risk_increase  min_risk  \
0            avg_cpu_temp  (53.28, 55.978]           0.001757  0.010097   
1           max_cpu_usage  (20.22, 24.381]           0.040850  0.001316   
2  avg_memory_utilization    (25.1, 38.93]           0.172135  0.000000   
3             uptime_days  (18.08, 25.916]           0.117793  0.000436   
4          memory_size_gb     (16.0, 32.0]           0.224655  0.001605   

   max_risk  
0  0.264706  
1  0.174429  
2  0.296752  
3  0.345917  
4  0.253680  
